In [31]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Linear(nn.Module):
    def __init__(self, in_dim, out_dim):
        super(Linear, self).__init__()
        self.fc = nn.Linear(in_dim, out_dim)
    
    def forward(self, x):
        return self.fc(x)
    
class MoELayer(nn.Module):
    def __init__(self, num_experts, in_dim, out_dim):
        super(MoELayer, self).__init__()
        self.num_experts = num_experts
        self.experts = nn.ModuleList([Linear(in_dim, out_dim) for _ in range(num_experts)])
        self.gate = nn.Linear(in_dim, num_experts) # [in_dim,num_experts]
    
    def forward(self, x):
        '''
        x [B,in_dim]
        return [B,out_dim]
        '''
        gate_scores = F.softmax(self.gate(x), dim=-1) # [B,num_experts]
        expert_outputs = torch.stack([expert(x) for expert in self.experts], dim=1) # [B,num_experts,out_dim]
        # 对每个样本，把所有expert的输出结果根据gate求和
        output = torch.bmm(gate_scores.unsqueeze(1), expert_outputs).squeeze(1) # [B,out_dim]
        return output

input_size = 5
output_size = 3
num_experts = 4
batch_size = 10

model = MoELayer(num_experts, input_size, output_size)

demo = torch.randn(batch_size, input_size)

output = model(demo)

print(output.shape)  # 输出: torch.Size([10, 3])

torch.Size([10, 3])


上面的gate和expert都是由一层线性层实现的

在此之上，可以设计更复杂的gate和expert

https://zhuanlan.zhihu.com/p/676980004


In [32]:
# Define the gate model 
class Gate(nn.Module): 
 def __init__(self, input_dim, 
                 num_experts, dropout_rate=0.1): 
    super(Gate, self).__init__() 

 # Layers 
    self.layer1 = nn.Linear(input_dim, 128) 
    self.dropout1 = nn.Dropout(dropout_rate) 

    self.layer2 = nn.Linear(128, 256) 
    self.leaky_relu1 = nn.LeakyReLU() 
    self.dropout2 = nn.Dropout(dropout_rate) 

    self.layer3 = nn.Linear(256, 128) 
    self.leaky_relu2 = nn.LeakyReLU() 
    self.dropout3 = nn.Dropout(dropout_rate) 

    self.layer4 = nn.Linear(128, num_experts) 

 def forward(self, x): 
    '''
    x: [B,input_dim]
    return gate [B,num_experts]
    '''
    x = torch.relu(self.layer1(x)) 
    x = self.dropout1(x) 

    x = self.layer2(x) 
    x = self.leaky_relu1(x) 
    x = self.dropout2(x) 

    x = self.layer3(x) 
    x = self.leaky_relu2(x) 
    x = self.dropout3(x) 

    return torch.softmax(self.layer4(x), dim=1)


In [33]:
class Expert(nn.Module): 
 def __init__(self, input_dim, hidden_dim, output_dim): 
        super(Expert, self).__init__() 
        self.layer1 = nn.Linear(input_dim, hidden_dim) 
        self.layer2 = nn.Linear(hidden_dim, output_dim) 

 def forward(self, x): 
    ''' 
    x: [B,input_dim]
    return  [B,output_dim]
    '''    
    x = torch.relu(self.layer1(x)) 
    return torch.softmax(self.layer2(x), dim=1)

In [34]:
class MoE(nn.Module): 
 def __init__(self, experts): 
       '''
       experts: List of Experts instance,trained/untrained
       '''
       super(MoE, self).__init__() 
       self.experts = nn.ModuleList(experts) 
       num_experts = len(experts) 
       # Assuming all experts have the same input dimension 
       input_dim = experts[0].layer1.in_features 
       self.gate = Gate(input_dim, num_experts) 

 def forward(self, x): 
       ''' 
       x: [B,in_dim]
       return [B,out_dim]
       '''
       gate_scores = F.softmax(self.gate(x), dim=-1) # [B,num_experts]
       expert_outputs = torch.stack([expert(x) for expert in self.experts], dim=1) # [B,num_experts,out_dim]
       # 对每个样本，把所有expert的输出结果根据gate求和
       output = torch.bmm(gate_scores.unsqueeze(1), expert_outputs).squeeze(1) # [B,out_dim]
       return output

       # another inplementation
       # # Get the weights from the gate network 
       # gate_scores = self.gate(x) # [B,num_experts]
       # # Calculate the expert outputs 
       # outputs = torch.stack([expert(x) for expert in self.experts], dim=2) #[B,out_dim,num_experts]
       # # Adjust the weights tensor shape to match the expert outputs 
       # gate_scores = gate_scores.unsqueeze(1).expand_as(outputs)  #[B,1,num_experts] expand to [B,out_dim,num_experts] 
       # # Multiply the expert outputs with the weights and 
       # # sum along the third dimension 
       # return torch.sum(outputs * gate_scores, dim=2) #[B,out_dim]

创建了一个合成数据集，其中包含三个类标签——0、1和2。基于类标签对特征进行操作，从而在数据中引入一些模型可以学习的结构。

数据被分成针对个别专家的训练集、MoE模型和测试集。我们确保专家模型是在一个子集上训练的，这样第一个专家在标签0和1上得到很好的训练，第二个专家在标签1和2上得到更好的训练，第三个专家看到更多的标签2和0。

每个专家使用基本的训练循环在不同的数据子集上进行单独的训练。循环迭代指定数量的epoch。

In [35]:
# Generate the dataset 
num_samples = 5000 
input_dim = 4 
hidden_dim = 32 
output_dim = 3

# Generate equal numbers of labels 0, 1, and 2 
y_data = torch.cat([ 
    torch.zeros(num_samples // 3), 
    torch.ones(num_samples // 3), 
    torch.full((num_samples - 2 * (num_samples // 3),), 2)  # Filling the remaining to ensure exact num_samples 
]).long() #[B,1]

# Biasing the data based on the labels 
x_data = torch.randn(num_samples, input_dim)    # [B,input_dim]
for i in range(num_samples): 
 if y_data[i] == 0: 
        x_data[i, 0] += 1  # Making x[0] more positive 
 elif y_data[i] == 1: 
        x_data[i, 1] -= 1  # Making x[1] more negative 
 elif y_data[i] == 2: 
        x_data[i, 0] -= 1  # Making x[0] more negative 

# Shuffle the data to randomize the order 
indices = torch.randperm(num_samples) 
x_data = x_data[indices] 
y_data = y_data[indices] 

# Verify the label distribution 
y_data.bincount() 

# Shuffle the data again
shuffled_indices = torch.randperm(num_samples) 
x_data = x_data[shuffled_indices] 
y_data = y_data[shuffled_indices] 

# Splitting data for training individual experts 
# Use the first half samples for training individual experts 
x_train_experts = x_data[:int(num_samples/2)] 
y_train_experts = y_data[:int(num_samples/2)] 

mask_expert1 = (y_train_experts == 0) | (y_train_experts == 1) 
mask_expert2 = (y_train_experts == 1) | (y_train_experts == 2) 
mask_expert3 = (y_train_experts == 0) | (y_train_experts == 2) 

# Select an almost equal number of samples for each expert 
num_samples_per_expert = min(mask_expert1.sum(), mask_expert2.sum(), mask_expert3.sum()) 

x_expert1 = x_train_experts[mask_expert1][:num_samples_per_expert] 
y_expert1 = y_train_experts[mask_expert1][:num_samples_per_expert] 

x_expert2 = x_train_experts[mask_expert2][:num_samples_per_expert] 
y_expert2 = y_train_experts[mask_expert2][:num_samples_per_expert] 

x_expert3 = x_train_experts[mask_expert3][:num_samples_per_expert] 
y_expert3 = y_train_experts[mask_expert3][:num_samples_per_expert] 

# Splitting the next half samples for training MoE model and for testing 
x_remaining = x_data[int(num_samples/2)+1:] 
y_remaining = y_data[int(num_samples/2)+1:] 

split = int(0.8 * len(x_remaining)) 
x_train_moe = x_remaining[:split] 
y_train_moe = y_remaining[:split] 

x_test = x_remaining[split:] 
y_test = y_remaining[split:] 

print(x_train_moe.shape,"\n", 
       x_test.shape,"\n", 
       x_expert1.shape,"\n", 
       x_expert2.shape,"\n",
       x_expert3.shape)

torch.Size([1999, 4]) 
 torch.Size([500, 4]) 
 torch.Size([1655, 4]) 
 torch.Size([1655, 4]) 
 torch.Size([1655, 4])


模型初始化和训练设置:

In [36]:
import torch.optim as optim
# Define hidden dimension 

# input_dim = 4 
# hidden_dim = 32 
# output_dim = 3

epochs = 500 
learning_rate = 0.001 

# Instantiate the experts 
expert1 = Expert(input_dim, hidden_dim, output_dim) 
expert2 = Expert(input_dim, hidden_dim, output_dim) 
expert3 = Expert(input_dim, hidden_dim, output_dim) 

# Set up loss 
criterion = nn.CrossEntropyLoss() 

# Optimizers for experts 
optimizer_expert1 = optim.Adam(expert1.parameters(), lr=learning_rate) 
optimizer_expert2 = optim.Adam(expert2.parameters(), lr=learning_rate) 
optimizer_expert3 = optim.Adam(expert3.parameters(), lr=learning_rate)

In [37]:
import copy
expert1_raw = copy.deepcopy(expert1)
expert2_raw = copy.deepcopy(expert2)
expert3_raw = copy.deepcopy(expert3)

In [38]:
print(expert1.state_dict().keys())
# 打印expert1第1个linear层的前三行权重
print(expert1.layer1.in_features) 
print(expert1.layer1.weight[:3,:])   

# expert1_raw 是 expert1 的深拷贝，所以权重值完全相同
# 使用all()函数检查张量中是否所有元素都为True
# 使用any()函数检查张量中是否至少有一个元素为True
assert (expert1_raw.layer1.weight == expert1.layer1.weight).all()

odict_keys(['layer1.weight', 'layer1.bias', 'layer2.weight', 'layer2.bias'])
4
tensor([[ 0.0015, -0.2619,  0.1639, -0.1806],
        [ 0.0024, -0.3892, -0.2939, -0.4964],
        [-0.0060, -0.1915,  0.0523, -0.0748]], grad_fn=<SliceBackward0>)


训练步骤

每个专家使用基本的训练循环在不同的数据子集上进行单独的训练。循环迭代指定数量的epoch。

In [39]:
# Training loop for expert 1 
for epoch in range(epochs): 
    optimizer_expert1.zero_grad() 
    outputs_expert1 = expert1(x_expert1) 
    loss_expert1 = criterion(outputs_expert1, y_expert1) 
    loss_expert1.backward() 
    optimizer_expert1.step() 

# Training loop for expert 2 
for epoch in range(epochs): 
    optimizer_expert2.zero_grad() 
    outputs_expert2 = expert2(x_expert2) 
    loss_expert2 = criterion(outputs_expert2, y_expert2) 
    loss_expert2.backward() 
    optimizer_expert2.step() 

# Training loop for expert 3 
for epoch in range(epochs): 
    optimizer_expert3.zero_grad() 
    outputs_expert3 = expert3(x_expert3) 
    loss_expert3 = criterion(outputs_expert3, y_expert3) 
    loss_expert3.backward()

MoE模型是由先前训练过的专家创建的，然后在单独的数据集上进行训练。训练过程类似于单个专家的训练，但现在门控网络的权值在训练过程中更新。

In [40]:
# Create the MoE model with the trained experts 
moe_model = MoE([expert1, expert2, expert3]) 

# Train the MoE model 
optimizer_moe = optim.Adam(moe_model.parameters(), lr=learning_rate) 
for epoch in range(epochs): 
    optimizer_moe.zero_grad() 
    outputs_moe = moe_model(x_train_moe) 
    loss_moe = criterion(outputs_moe, y_train_moe) 
    loss_moe.backward() 
    optimizer_moe.step()

In [41]:
# 经过每个专家单独训练后，expert1 更新，expert1_raw 没有
# 因为 expert1_raw 是 expert1 的深拷贝
print(expert1.layer1.weight[:3,:])  
print(expert1_raw.layer1.weight[:3,:])  

tensor([[-0.1427, -0.6583, -0.0514,  0.1758],
        [-0.4213, -0.9130, -0.1952, -0.8314],
        [-0.5328, -0.5751,  0.0824,  0.6536]], grad_fn=<SliceBackward0>)
tensor([[ 0.0015, -0.2619,  0.1639, -0.1806],
        [ 0.0024, -0.3892, -0.2939, -0.4964],
        [-0.0060, -0.1915,  0.0523, -0.0748]], grad_fn=<SliceBackward0>)


在开始，拷贝一份没训练过的expert

In [42]:
expert1_raw_copy = copy.deepcopy(expert1_raw)
expert2_raw_copy = copy.deepcopy(expert2_raw)
expert3_raw_copy = copy.deepcopy(expert3_raw)

In [43]:

# Create the MoE model with the un-trained experts 
moe_model_raw = MoE([expert1_raw, expert2_raw, expert3_raw]) 

# Train the MoE model 
optimizer_moe_raw = optim.Adam(moe_model_raw.parameters(), lr=learning_rate) 
for epoch in range(epochs): 
    optimizer_moe_raw.zero_grad() 
    outputs_moe_raw = moe_model_raw(torch.cat([x_train_experts,x_train_moe])) 
    loss_moe_raw = criterion(outputs_moe_raw, torch.cat([y_train_experts,y_train_moe])) 
    loss_moe_raw.backward() 
    optimizer_moe_raw.step()

In [44]:
# expert1和expert1_raw都更新了
# expert1_raw作为参数传入moe_raw时是浅拷贝，值会被更改
print(expert1.layer1.weight[:3,:])  
print(expert1_raw.layer1.weight[:3,:])  
print(expert1_raw_copy.layer1.weight[:3,:])

tensor([[-0.1427, -0.6583, -0.0514,  0.1758],
        [-0.4213, -0.9130, -0.1952, -0.8314],
        [-0.5328, -0.5751,  0.0824,  0.6536]], grad_fn=<SliceBackward0>)
tensor([[-0.2008, -0.4716, -0.0604,  0.0981],
        [-0.1919, -0.5522, -0.2184, -0.4029],
        [-0.2796, -0.3123,  0.0833,  0.4454]], grad_fn=<SliceBackward0>)
tensor([[ 0.0015, -0.2619,  0.1639, -0.1806],
        [ 0.0024, -0.3892, -0.2939, -0.4964],
        [-0.0060, -0.1915,  0.0523, -0.0748]], grad_fn=<SliceBackward0>)


In [45]:
# Evaluate all models 
def evaluate(model, x, y): 
    '''
    x [B,in_dim]
    y [B]
    '''
    with torch.no_grad(): 
        outputs = model(x) # [B,out_dim]
        _, predicted = torch.max(outputs, 1) #[B] [B]
        correct = (predicted == y).sum().item() 
        accuracy = correct / len(y) 
    return accuracy

accuracy_expert1 = evaluate(expert1, x_test, y_test) 
accuracy_expert2 = evaluate(expert2, x_test, y_test) 
accuracy_expert3 = evaluate(expert3, x_test, y_test) 
accuracy_moe = evaluate(moe_model, x_test, y_test) 
accuracy_moe_raw = evaluate(moe_model_raw,x_test,y_test)

print("Expert 1 Accuracy:", accuracy_expert1) 
print("Expert 2 Accuracy:", accuracy_expert2) 
print("Expert 3 Accuracy:", accuracy_expert3) 
print("Mixture of trained-Experts Accuracy:", accuracy_moe) 
print("Mixture of untrained-Experts Accuracy:", accuracy_moe_raw) 

Expert 1 Accuracy: 0.68
Expert 2 Accuracy: 0.656
Expert 3 Accuracy: 0.674
Mixture of trained-Experts Accuracy: 0.674
Mixture of untrained-Experts Accuracy: 0.684


每次前向传播都要计算一遍所有专家的输出？

Switch Transformer讲了比较多有关于节省计算资源的，说明为什么可以用更少的资源实现更大参数量的模型。前向传播只需要计算TopK专家的输出，

无论 x 是 [B,in_dim] 还是 [B,S,in_dim]，都是形如[in_dim]的token作为为基本单元，每个token对应num_experts个gate_scores和num_experts个expert的输出，因此以下把x的注释标注为 [...,in_dim], `...`中的维度任意

为了实现在前向传播只计算TopK专家的输出，把`Gate`更新为`TopKrouter`，使用`TopKrouter`的称为 `sparseMoE`，计算时间更短。

参考：https://cloud.tencent.com/developer/article/2391130

In [88]:
class TopKRouter(nn.Module):
    def __init__(self, input_dim, num_experts, top_k):
        super(TopKRouter,self).__init__()
        self.linear = nn.Linear(input_dim, num_experts)
        self.top_k = top_k

    def forward(self, x):
        '''
            x [...,in_dim]
        return 
            router_output [...,num_experts]
            top_k_indices [...,top_k]
        '''
        # 计算每个专家的分数
        logits = self.linear(x)
        # 获取Top-K分数和对应的索引
        top_k_logits, top_k_indices = logits.topk(self.top_k, dim=-1)
        # 没route到的expert的logits会被置为-inf,softmax之后变为0
        zeros = torch.full_like(logits, float('-inf'))
        sparse_logits = zeros.scatter(-1, top_k_indices, top_k_logits)
        router_output = F.softmax(sparse_logits, dim=-1)
        return router_output, top_k_indices

#Testing this out:
num_experts = 5
top_k = 2
input_dim = 32

x = torch.randn(2, 4, input_dim)  # Example input [B,S,n_embed]
top_k_gate = TopKRouter(input_dim, num_experts, top_k)
gate_scores, top_k_indices = top_k_gate(x)    # [B,S,num_experts]  [B,S,top_k]
gate_scores.shape, gate_scores, top_k_indices
#And it works!!

(torch.Size([2, 4, 5]),
 tensor([[[0.4668, 0.0000, 0.0000, 0.0000, 0.5332],
          [0.0000, 0.0000, 0.0000, 0.5466, 0.4534],
          [0.3514, 0.0000, 0.0000, 0.0000, 0.6486],
          [0.0000, 0.0000, 0.6186, 0.3814, 0.0000]],
 
         [[0.0000, 0.0000, 0.4190, 0.0000, 0.5810],
          [0.0000, 0.7759, 0.0000, 0.0000, 0.2241],
          [0.4466, 0.5534, 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.4559, 0.5441, 0.0000]]],
        grad_fn=<SoftmaxBackward0>),
 tensor([[[4, 0],
          [3, 4],
          [4, 0],
          [2, 3]],
 
         [[4, 2],
          [1, 4],
          [1, 0],
          [3, 2]]]))

In [102]:

class SparseMoE(nn.Module):
    def __init__(self,experts,top_k):
        super(SparseMoE,self).__init__()
        self.num_experts = len(experts)
        self.top_k = top_k
        # suppose all experts share same structure
        self.input_dim = experts[0].layer1.in_features
        self.output_dim = experts[0].layer2.out_features
        
        self.router = TopKRouter(self.input_dim, self.num_experts, top_k)
        self.experts = nn.ModuleList(experts)

    def forward(self, x):
        '''
        x [...,in_dim] take [B,S,in_dim] as example
        return [...,out_dim]
        '''

        output = torch.zeros(*x.shape[:-1], self.output_dim)  #[B,S,out_dim]
        
        gate_scores, indices = self.router(x) #[B,S,num_experts]  [B,S,top_k]
        
        # Reshape inputs for batch processing
        flat_x = x.view(-1, x.size(-1))  # [N, in_dim] where N = B*S*...
        flat_gate_scores = gate_scores.view(-1, gate_scores.size(-1)) # [N,num_experts]
        flat_output =  output.view(-1, output.size(-1)) # [N, out_dim]

        # Process each expert in parallel
        for i, expert in enumerate(self.experts):
            # Create a mask for the inputs where the current expert should consider
            mask = (indices == i).any(dim=-1)   #[B,S]
            flat_mask = mask.view(-1)           #[N]

            if flat_mask.any():
                expert_input = flat_x[flat_mask]     # [num_selected, in_dim]
                expert_output = expert(expert_input) # [num_selected, out_dim]

                # Extract and apply gating scores
                expert_gate_scores = flat_gate_scores[flat_mask, i].unsqueeze(-1)   # [num_selected,1]
                weighted_output = expert_output * expert_gate_scores  # [num_selected,out_dim]

                # Update final output additively by indexing and adding
                flat_output[flat_mask] += weighted_output

        return flat_output.view(*x.shape[:-1], self.output_dim)  #[B,S,out_dim]


In [103]:
moe_model_sparse = SparseMoE([expert1_raw_copy,expert2_raw_copy,expert3_raw_copy],top_k = 2)

optimizer_moe_sparse = optim.Adam(moe_model_sparse.parameters(),lr=learning_rate)
for epoch in range(epochs):
    optimizer_moe_sparse.zero_grad()
    outputs_moe_sparse = moe_model_sparse(torch.cat([x_train_experts,x_train_moe]))
    loss = criterion(outputs_moe_sparse,torch.cat([y_train_experts,y_train_moe]))
    loss.backward()
    optimizer_moe_sparse.step()


可以看到效果比moe_model_raw稍微差一点，但比其他模型效果都要更好，并且计算效率最高，训练时间最短

In [104]:
accuracy_moe_sparse = evaluate(moe_model_sparse,x_test,y_test)
print("Mixture of Sparse untrained-Experts Accuracy:", accuracy_moe_sparse) 

Mixture of Sparse untrained-Experts Accuracy: 0.682


5000个样本，其中1667个标签为0，1667个标签为1，1667个标签为1，1666个标签为2

5000个里面划分出2500和2500个样本，

- 前一个2500为 x_train_expert，从找出 1665个label为0或1，1665个label为0或2，1665个label为1或2，x_expert
- 后一个2500拆分为2000 x_train_moe 和500 x_test

不同的moe_model
- moe_model: mixture of pretrained experts，每个expert先在各自的x_expert上训练，然后一起在x_train_moe个样本上训练，每个token计算所有expert的输出平均，主要学习gate
- moe_model_raw: mixture of untrained experts，直接一起在x_train_expert+x_train_moe个样本上训练，每个token计算所有expert的输出平均，同时学习gate和expert，耗时最长
- moe_model_sparse：mixture of sparse untrained experts，每个token只计算top_k个expert的输出平均，一起在x_train_expert+x_train_moe个样本上训练，同时学习router和expert，计算效率最高